<p class="h1">ECE 447 - Notebook 09</p>
<p class="h2">Logistic Regression</p>

In this section, we cover the following concepts:

- logistic regression for classification;
- performance metrics (confusion matrix, ROC curves);
- binary and multi class classification;

# Logistic regression

**Classification** is a task associated wih supervised learning. The developed model is based on previous information and able to classify new data by choosing the class that is most suitable for a given data point.

One of the methods to train a model for classification purposes is *logistic regression*. Logistic regression is a classification method used to assign observations to a discrete set of classes. The method is similar to linear regression with the difference that, instead of outputing continuous values, logistic regression generates outputs over a discrete set of values (classes, categories). To determine the classes, logistic regression uses a function (sigmoid, softmax) to map a probabilistic output to two or more discrete classes.


Let us compare the linear regression to the logistic one.

- **Linear Regression** makes predictions over a continuous range of values (numbers in a range), such as the value of a house based on the area, number of rooms, baths, location, etc. 
- **Logistic Regression** makes predictions over a discrete set of values (only specific values or categories), such as the type of wine based on the grapes used, the region, color, etc.

There are three types of **logistic regression**.

- Binary: only two possibilities. For example, Accepted/Rejected,  Benign/Malignant.
- Multi: More than two possibilities. For example, Malbec, Cabernet Sauvignon, Merlot, Syrah.
- Ordinal: Low, Medium, High.

In [ ]:
import numpy as np
import pandas as pd
import random
import math

import matplotlib.pyplot as plt

A simple function to create samples in two categories

In [ ]:
def gen_logistic_samples(n, sz=20, a=0.7, s=5, b=5):
    """
    n: Number of data points to generate
    sz: Range used to generate [0,sz]
    a: Slope
    s: Spread
    b: Intercept with y axis
    
    Returns:
      X0,X1: Data points 
      Y: Category for each point
    """
    X0 = sz * np.random.ranf(n)
    Yo = a*X0 + b
    X1 = Yo + 2*s*np.random.ranf(n) - s
    Y = np.ones(n)
    for i in range(n):
        if X1[i]<b:
            Y[i] = 0
        if b< X1[i] < (b+s/3):
            Y[i] = random.randint(0,1)
    return X0,X1,Y

In [ ]:
x1,x2,y = gen_logistic_samples(100, sz=50, s=30, a=0.4, b=20)

In [ ]:
import pandas as pd
pd.DataFrame({'x1':x1,'x2':x2,'Y':y}).head(10)

In [ ]:
plt.scatter(x1[y==1],x2[y==1],label="Y=1")
plt.scatter(x1[y==0],x2[y==0],label="Y=0")
plt.legend();

Logistic regression uses a function to determine the probability that an output data point belongs to one category or the other. One of those functions is the *sigmoid* function, $\sigma(z)$:

$$
\sigma(z) = \frac{1}{1+e^{-z}}
$$

In [ ]:
def sigmoid(z):
    return 1/(1+math.e**-z)

The sigmoid function determines a probability that the input feature vector produces a valid observation. Expressed in terms of conditional probability, we want to compute $P(Y=1|X=x)$. The probability of the observation belongs to the category $Y=1$, given the feature vector $X=x$.

The prediction function $P(Y=1|X=x)$ returns a probability value between 0 and 1. To map this probabilities to a discrete class, we need to define a *threshold* value that will determine our decision boundary. Values above the decision boundary are in one class (class 1), and values below in another class (class 0).

$$
\textrm{Class 1}\quad \textrm{if } P(Y=1|X=x)\ge 0.5\\
\textrm{Class 0}\quad \textrm{if } P(Y=1|X=x)< 0.5
$$

In [ ]:
xl = np.linspace(-10, 10, 20)
plt.plot(xl,sigmoid(xl))
plt.plot(xl,0.5*np.ones(xl.size),'r',label='Decision')
plt.grid()
plt.legend();

In [ ]:
def decision(prob, boundary=0.5):
    if prob >= boundary:
        return 1
    return 0

Using the sigmoid function, $\sigma(z)$, and the decision boundary, we can define a prediction function for logistic regression. The prediction function, in this case, returns the probability of the observation to be one class, ($P(Y=1|X=x)$, or $P(\textrm{Class=1})$). Probabilities closer to 1 show that our model is more confident that the observation is in class 1.

Our hypothesis/prediction for two-feature data is as follows: 

$$
h_\theta(\tilde{x}) = h_\theta(x_1,x_2) = \sigma(\theta_0 + \theta_1x_1 + \theta_2x_2).
$$

Replacing and defining it in terms of probabilities (returns a probability value between 0 and 1), we have:

$$
P(\textrm{Class=1}) = \frac{1}{1+e^{-h_\theta(\tilde{x})}}.
$$

In [ ]:
def predict(Features, Weights):
    z = Features @ Weights
    return sigmoid(z)

# Cost function

The sigmoid function transforms our prediction into a non-linear function. Since the hypothesis is non-linear, we can not use the Mean of Square Error as a cost function. Instead of Mean Squared Error, we use a cost function called **Cross-Entropy** or **Log Loss**. 

**Cross-entropy loss** can be divided into two separate cost functions: one for $Y=1$, and one for $Y=0$.

$$
J(\theta) =\frac{1}{m}\sum_{i=1}^m Cost(h_\theta(\tilde{x_i}),y_i)
$$

$$
\begin{array}{}
Cost(h_\theta(\tilde{x_i}),y_i) = -\log(h_\theta(\tilde{x})) & \textrm{if } y=1\\
Cost(h_\theta(\tilde{x_i}),y_i) = -\log(1-h_\theta(\tilde{x})) & \textrm{if } y=0
\end{array}
$$

Taking a logarithm over $h_{\theta}(\tilde{x})$ makes the cost function a smooth monotonic function. It enables minimization of the cost using gradient descent.

In [ ]:
x = np.linspace(0,1,20)
plt.figure(figsize=(10, 3))
ax = plt.subplot(1,2,1)
plt.plot(x,-np.log(sigmoid(x)))
ax.set_title('y=1  ($-\log(S(x))$)')
ax = plt.subplot(1,2,2)
plt.plot(x,-np.log(1-sigmoid(x)))
ax.set_title('y=0  ($-\log(1-S(x))$)');

## Cross-Entropy  (Log Loss)

Let us consider that we have two classes (y=1 and y=0). We define the following **Cross-Entropy** cost function:
$$
J(\theta)=-\frac{1}{m}\sum_{i=1}^m[y_i\log(h_\theta(\tilde{x_i})) + (1-y_i)\log(1-h_\theta(\tilde{x_i}))).
$$

Multiplying by $y$ and $1-y$ allows the cost function to select which part of the cost function to use based on y=1 or y=0.

A matrix representation of the cost function is as follows:
$$
J(\theta)=\frac{1}{m}[-Y^T\log(h_\theta(X)) - (1-Y)^T\log(1-h_\theta(X)].
$$

In [ ]:
def cost_function(Features, Labels, Weights):
    m = len(Labels)
    Ypred = predict(Features, Weights)
    # cost when label=1
    cost_class1 = -Labels * np.log(Ypred)
    # cost when label=0
    cost_class2 = -(1 - Labels) * np.log(1 - Ypred)
    tot_cost = cost_class1 + cost_class2
    tot_cost = tot_cost.sum()/m
    return tot_cost

With a new cost function, we need to modify our gradient descendant calculation: 

$$
\nabla J(\theta) = X.(S(h_\theta)-Y).
$$

In [ ]:
def update_weights(Features, Labels, Weights, learning_rate):
    m = len(Features)
    Ypred = predict(Features, Weights)
    gradient = Features.T @ (Ypred - Labels)
    newWeights = Weights - learning_rate *(1/m) * gradient
    return newWeights

The following function makes predictions based on the specifice decision boundary:

In [ ]:
def classify(Ypreds, boundary=0.5):
    decision_bound = np.vectorize(decision)
    return decision_bound(Ypreds, boundary).flatten()

In [ ]:
P = [0.23, 0.56, 0.45, 0.74]
print(P)
classify(P)

The `train` function is similar to the train function used for linear regression:

In [ ]:
def train(Features, Labels, learning_rate, n_iterations):
    Weights = np.zeros((Features.shape[1],1))
    cost_history = []
    for i in range(n_iterations):
        Weights = update_weights(Features, Labels, Weights, learning_rate)
        cost = cost_function(Features, Labels, Weights)
        cost_history.append(cost)
    return Weights, cost_history

Now we have all the components needed to create a logistic regression model for our example problem.

In [ ]:
plt.scatter(x1[y==1],x2[y==1],label="Y=1")
plt.scatter(x1[y==0],x2[y==0],label="Y=0")
plt.legend();

In [ ]:
bias = np.ones(len(x1))
Features = np.array([bias, x1, x2]).T
Labels = y.reshape(len(y),1)

In [ ]:
w,h = train(Features, Labels, 0.005, 50000)

In [ ]:
w

In [ ]:
plt.plot(h)
plt.title('Cost history');

In [ ]:
print("Final cost function:",h[-1])

In [ ]:
Yp = predict(Features,w).T

In [ ]:
Yp.flatten()[:20]

In [ ]:
Yp = classify(Yp)

In [ ]:
Yp[:20]

In [ ]:
coef = w[1:]
intercept = w[0]
def line(x):
    return (-(x * coef[0]) - intercept) / coef[1]
    
plt.figure(figsize=(10, 3))
ax = plt.subplot(1,2,1)
ax.scatter(x1[Yp==1],x2[Yp==1])
ax.scatter(x1[Yp==0],x2[Yp==0])
plt.plot([0, 50], [line(0), line(50)],color='r')
ax.set_title('Prediction')
ax = plt.subplot(1,2,2)
ax.scatter(x1[y==1],x2[y==1])
ax.scatter(x1[y==0],x2[y==0])
ax.set_title('Ground truth');

# Case Study

Let us create a classification model for the advertising data.

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('data/Advertising.csv')

In [ ]:
data.head()

We add a new categorical feature `label` to indicate whether a sale was higher than $15K or not. 

In [ ]:
labels = np.ones(shape=(len(data),1))
labels[data['sales']<15] = 0
lbl = pd.DataFrame(labels, columns=['label'])
res = pd.concat([data, lbl], axis=1)

In [ ]:
res.head()

In [ ]:
y = res['sales'].values
cat = res['label'].values
x1 = res['TV'].values
x2 = res['radio'].values

In [ ]:
plt.scatter(x1[cat==1],x2[cat==1], label='>=$15K')
plt.scatter(x1[cat==0],x2[cat==0], label='<$15K')
plt.xlabel('TV')
plt.ylabel('radio')
plt.legend();

In [ ]:
# Setup imput data
Labels = cat.reshape(len(cat),1)

bias = np.ones(len(x1))
Features = np.array([bias, x1, x2]).T

In [ ]:
%%time
w,h = train(Features, Labels, 0.0001, 500000)

In [ ]:
w.T

In [ ]:
plt.plot(h)
plt.title('Cost history');

In [ ]:
print("Final cost function:",h[-1])

In [ ]:
Yp = predict(Features,w).T
Yp = classify(Yp)

In [ ]:
plt.figure(figsize=(10,5))
plt.subplot(121)
plt.title("Ground Truth")
plt.scatter(x1[cat==1],x2[cat==1])
plt.scatter(x1[cat==0],x2[cat==0])
plt.xlabel('TV')
plt.ylabel('radio')
plt.ylim(-1,52)

plt.subplot(122)
plt.title("Prediction")
plt.scatter(x1[Yp==1],x2[Yp==1])
plt.scatter(x1[Yp==0],x2[Yp==0])
plt.xlabel('TV')
plt.ylabel('radio')
plt.ylim(-1,52)

coef = w[1:]
intercept = w[0]
def line(x):
    return (-(x * coef[0]) - intercept) / coef[1]  
plt.plot([0, 300], [line(0), line(300)],color='r')

<div class="alert alert-success">
<h2>Exercise</h2>

✏️ Using the Advertisement data set, train two models one with TV and Newspaper, and other with radio and newspaper as features. 
</div>

# Normalization and standardization

Let us apply the processes of data normalization and scaling to the features, and compare the results.

In [ ]:
def standardization(Features):
    """
    Take a set of features return normalize values using
    the mean and standard deviation of each feature
    """
    mean=np.mean(Features,axis=0)
    std=np.std(Features,axis=0)
    Features_zss = (Features - mean)/std
    return Features_zss

In [ ]:
def normalization(Features):
    """
    Take a set of features return normalize values using
    (x -xmin) / (xmax – xmin) of each feature
    """
    fmin = np.min(Features,axis=0)
    fmax = np.max(Features,axis=0)
    Features_norm = (Features - fmin)/(fmax-fmin)
    return Features_norm

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve, auc
cm = confusion_matrix(cat, Yp)
pd.DataFrame(cm)

In [ ]:
tn,fp,fn,tp = cm.flatten()

In [ ]:
print("Model accuracity:",(tn+tp)/200)
print("Misclassified:",(fn+fp)/200)
print("% of error of true No:",fn/(fn+tn)*100)
print("% of error of true Yes:",fp/(fp+tp)*100)

In [ ]:
print("Model accuracity:",accuracy_score(cat,Yp))

In [ ]:
Yprob = predict(Features,w).flatten()
# Compute ROC curve and ROC area
fpr, tpr, _ = roc_curve(cat, Yprob, pos_label=1)
roc_auc = auc(fpr, tpr)

In [ ]:
plt.plot(fpr, tpr, color='darkorange', label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right")
plt.show();

<div class="alert alert-success">
<h2>Exercise</h2>

✏️ Compare the results using *normalization*, *standardization* and the results obtained with no scaling.
</div>

## Logistic regression with scikit-learn

Scikit-learn provides a module to compute logistic regression with different types of approaches (solvers) to find the minimum of the cost function (alternatives to gradient descendant).

The current supported solvers are: “liblinear”, “newton-cg”, “lbfgs”, “sag” and “saga”

For more information about the different solvers and when they can be applied, see https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression.

In [ ]:
from sklearn import linear_model

In [ ]:
logreg = linear_model.LogisticRegression(solver='liblinear')

Recalling out Advertising data

In [ ]:
Features = np.array([x1, x2]).T

In [ ]:
logreg.fit(Features,cat)

In [ ]:
logreg.coef_,logreg.intercept_

In [ ]:
Ypre = logreg.predict(Features)

In [ ]:
plt.figure(figsize=(10,5))
plt.subplot(121)
plt.title("Ground Truth")
plt.scatter(x1[cat==1],x2[cat==1])
plt.scatter(x1[cat==0],x2[cat==0])
plt.xlabel('TV')
plt.ylabel('radio')
plt.ylim(-1,52)

plt.subplot(122)
plt.title("Prediction")
plt.scatter(x1[Yp==1],x2[Yp==1])
plt.scatter(x1[Yp==0],x2[Yp==0])
plt.xlabel('TV')
plt.ylabel('radio')
plt.ylim(-1,52)

coef = w[1:]
intercept = w[0]
def line(x):
    return (-(x * coef[0]) - intercept) / coef[1]  
plt.plot([0, 300], [line(0), line(300)],color='r')

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(cat, Ypre)
pd.DataFrame(cm)

In [ ]:
tn, fp, fn, tp = cm.flatten()
tot = np.sum(cm)

In [ ]:
print("Model accuracy:",(tn+tp)/tot)
print("Misclassified:",(fn+fp)/tot)
print("% of error of true No:",fn/(fn+tn)*100)
print("% of error of true Yes:",fp/(fp+tp)*100)

In [ ]:
print("Model accuracity:",accuracy_score(cat,Yp))

<div class="alert alert-success">
<h2>Exercise</h2>

✏️ Compare the classification behaviour of logistic regression (`LogisticRegression`) method using different solvers (`liblinear`, `newton-cg`, `lbfgs`, `sag` and `saga`). Verify whether changing `max_iter` parameter in the `LogisticRegression` constructor produces different results.
</div>

# Multi-categorical classification

So far, we have focused on a binary classification (two classes) problem. Quite often, we deal with problems that have more than 2 classes. In these cases, we can use a binary classification approach to create a prediction model.

For example, let us generate data points that are divided into 3 classes.

In [ ]:
from sklearn.datasets import make_blobs
X, y = make_blobs(n_samples=150, centers=3, n_features=2,random_state=2)

In [ ]:
plt.scatter(X[:,0][y==0],X[:,1][y==0],label='y=0')
plt.scatter(X[:,0][y==1],X[:,1][y==1],label='y=1')
plt.scatter(X[:,0][y==2],X[:,1][y==2],label='y=2')
plt.legend();

In [ ]:
bias = np.ones(len(X))
Features = np.vstack((bias,X.T)).T

## One vs All (One-vs-Rest)

The **One vs All (or One-vs-Rest)** approach uses a binary classification approach to create a model for multiclass classification. 

The general idea is to generate $k$ different classifiers, one for each class. Each classifier is created using a binary classification to separate the class we are interested in from the rest of classes. After we have $k$ classifiers, the final prediction is determined using the classifier that produces a higher probability.

We start by creating $k$ classification models, one for each class/category (3 models in our case).

### First category (y=0)

We assign categories to our observations; the observations we are interested in (y=0) are one class, while the rest of the observations constitute the other class.

In [ ]:
Labels = np.zeros(len(y))
Labels[y==0] = 1

In [ ]:
plt.scatter(X[:,0][Labels==1],X[:,1][Labels==1], label='y=0')
plt.scatter(X[:,0][Labels==0],X[:,1][Labels==0], label='y=1 & y=2')
plt.legend();

In [ ]:
Labels = Labels.reshape(len(y),1)

We train the model using out binary classification approach.

In [ ]:
%%time
w0,h0 = train(Features, Labels, 0.01, 8000)

In [ ]:
plt.plot(h0),h0[-1];

### Second category (y=1)

We repeat the same process, but this time we are 'focus' on the second class (y=1).

In [ ]:
Labels = np.zeros_like(y)
Labels[y==1] = 1

In [ ]:
plt.scatter(X[:,0][Labels==1],X[:,1][Labels==1], label='y=1')
plt.scatter(X[:,0][Labels==0],X[:,1][Labels==0], label='y=0 & y=2')
plt.legend();

In [ ]:
Labels = Labels.reshape(len(y),1)

And we train our second model.

In [ ]:
%%time
w1,h1 = train(Features, Labels, 0.0001, 10000)

In [ ]:
plt.plot(h1),h1[-1];

### Third category (y=2)

Finally, we repeat  the process for the last class (y=2).

In [ ]:
Labels = np.zeros_like(y)
Labels[y==2] = 1

In [ ]:
# plt.scatter(X[:,0],X[:,1],c=Labels);
plt.scatter(X[:,0][Labels==1],X[:,1][Labels==1], label='y=2')
plt.scatter(X[:,0][Labels==0],X[:,1][Labels==0], label='y=0 & y=1')
plt.legend();

In [ ]:
Labels = Labels.reshape(len(y),1)

In [ ]:
%%time
w2,h2 = train(Features, Labels, 0.001, 4000)

In [ ]:
plt.plot(h2),h2[-1];

### Final model

Now we have a model for each class.

In [ ]:
W = np.hstack((w0,w1,w2))

In [ ]:
W

The final classification takes all models and evaluates them to determine, for each observation, which model produces the higher probability. Based on that, an observation is classfied to the class for which the 'winning' model has been built. 

In [ ]:
def classify_multi(Features,W):
    Yp = np.apply_along_axis(lambda w: predict(Features,w),0,W)
    return np.argmax(Yp,axis=1)

In [ ]:
Yp

In [ ]:
Yp = classify_multi(Features, W)

In [ ]:
Yp

In [ ]:
plt.scatter(X[:,0],X[:,1],c=Yp);

In [ ]:
plt.scatter(X[:,0][y==0],X[:,1][y==0],label='y=0')
plt.scatter(X[:,0][y==1],X[:,1][y==1],label='y=1')
plt.scatter(X[:,0][y==2],X[:,1][y==2],label='y=2')
plt.legend();

In [ ]:
y

In [ ]:
Yp

We create a confusion matrix to determine the performance of our classification model.

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y, Yp)
pd.DataFrame(cm)

In [ ]:
tot = np.sum(cm)
diag = np.sum(np.diagonal(cm))

print("Model accuracity:",accuracy_score(y,Yp))
print("Misclassified: %{}".format((tot-diag)/tot*100))

We generate ROC curves for our model, one for each classifier.

In [ ]:
for i in range(3):
    Yprob = predict(Features,W[:,i]).flatten()
    fpr, tpr, _ = roc_curve(y, Yprob, pos_label=i)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label='ROC curve class=%d (area = %0.2f)' % (i,roc_auc))
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right")
plt.show();

# Non linear separable

For more complex classification problems, our linear classification approach does not work well. 

## Motivation example

Let us check the following example.

In [ ]:
from sklearn.datasets import make_gaussian_quantiles
X, y = make_gaussian_quantiles(n_samples=100, n_features=2, n_classes=2, random_state=2)

In [ ]:
plt.scatter(X[:,0][y==0],X[:,1][y==0])
plt.scatter(X[:,0][y==1],X[:,1][y==1]);

In [ ]:
bias = np.ones(len(X))
Features = np.vstack((bias,X.T)).T
Labels = y.reshape(len(y),1)

In [ ]:
%%time
w,h = train(Features, Labels, 0.001, 40000)

In [ ]:
plt.plot(h),h[-1];

In [ ]:
Yp = predict(Features,w)
Yp = classify(Yp)

We train our model and check the obtained predictions.

In [ ]:
plt.scatter(X[:,0],X[:,1],c=Yp);

In [ ]:
print("Model accuracity:",accuracy_score(y,Yp))

It is clear that our linear classification model does not perform well.

## Polynomial features

Similar to linear regression, we can create more complex models by using polynomial features.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
poly_features = PolynomialFeatures(degree = 2)

In [ ]:
X_poly = poly_features.fit_transform(X)

In [ ]:
%%time
w,h = train(X_poly, Labels, 0.0001, 200000)

In [ ]:
plt.plot(h), h[-1];

In [ ]:
Yp = predict(X_poly,w)
Yp = classify(Yp)

In [ ]:
plt.scatter(X[:,0],X[:,1],c=Yp);

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y, Yp)
pd.DataFrame(cm)

In [ ]:
print("Model accuracity:",accuracy_score(y,Yp))

We can see that we obtain a better model using a polynomial feature of degree 2. 

If we plot the decision boundary of our model, we obtain the following. 

In [ ]:
xg = np.linspace(X[:,0].min(), X[:,0].max(), 150)
yg = np.linspace(X[:,1].min(), X[:,1].max(), 150)

xx, yy = np.meshgrid(xg,yg)
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())
Xcont = poly_features.fit_transform(np.vstack((xx.ravel(),yy.ravel())).T)
Zp = predict(Xcont,w)
Z = classify(Zp).reshape(xx.shape)
plt.contour(xx,yy,Z,levels=1)
plt.scatter(X[:,0],X[:,1],c=Yp)
plt.title("Using our implementation");

<div class="alert alert-success">
<h2>Exercise</h2>

✏️ Use the *one vs rest* technique to create a classifier for the data generated by:
    
    X, y = make_gaussian_quantiles(n_samples=150, n_features=2, n_classes=3, cov=0.3,random_state=0).

Evaluate different polynomial degrees to determine the parameters that produce a good model. 
</div>

## Using scikit-learn

Logistic regression module in scikit-learn is part of the linear model modules.

In [ ]:
from sklearn import linear_model

`LogisticRegression` provides an option to deal with multi-class problems. You can specify it through the `multi_class` parameter. Options are `multimodal`, `ovr` (One-vs-Rest) and `auto`. The `multimodal` option uses a statistical method to do the classification. It applies the *softmax* function instead of the *sigmoid* function used by `ovr`. 

(See https://dataaspirant.com/2017/03/07/difference-between-softmax-function-and-sigmoid-function/ for more information about *softmax*). 

In [ ]:
logreg = linear_model.LogisticRegression(solver='lbfgs')

Let us reuse the data points from the previous example.

In [ ]:
X_poly = poly_features.fit_transform(X)
logreg.fit(X_poly,y)

In [ ]:
Yp = logreg.predict(X_poly)

In [ ]:
cm = confusion_matrix(y, Yp)
pd.DataFrame(cm)

In [ ]:
print("Model accuracity:",accuracy_score(y,Yp))

In [ ]:
xg = np.linspace(X[:,0].min(), X[:,0].max(), 150)
yg = np.linspace(X[:,1].min(), X[:,1].max(), 150)

xx, yy = np.meshgrid(xg,yg)
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())
Xcont = poly_features.fit_transform(np.vstack((xx.ravel(),yy.ravel())).T)
Z = logreg.predict(Xcont).reshape(xx.shape)
plt.contour(xx,yy,Z,levels=1)
plt.scatter(X[:,0],X[:,1],c=Yp)
plt.title("Using scikit-learn");

For the ROC curve, we have the following. 

In [ ]:
Yprob = logreg.predict_proba(X_poly)
# Compute ROC curve and ROC area
fpr, tpr, _ = roc_curve(y, Yprob[:,1], pos_label=1)
roc_auc = auc(fpr, tpr)

In [ ]:
plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right")
plt.show();

# Recap

In this section, we introduced logistic regression, a way to use linear regression to create a classification model. We showed how to use a binary classification approach to implement a multiclass classification model using the One-vs-Rest technique.